# Hello world example

Before start, configure an account in this platforms and get apikeys:
- LangSmith (LANGSMITH_API_KEY)
- LangFuse (LANGFUSE_PUBLICEY, LANGFUSE_SECRETKEY)
- OpenAI (OPENAI_API_KEY)
- SerpAPI (SERPAPI_API_KEY)

In [ ]:
! pip install -U langchain langchain_community langchain_mcp_adapters langfuse langchain_anthropic langchain-openai langchain-google-genai dotenv google-search-results arxiv

### Imports

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import Tool, tool
from langchain_community.utilities import SerpAPIWrapper
from langchain_community.utilities import ArxivAPIWrapper
from langchain_mcp_adapters.client import MultiServerMCPClient

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

# Logging with Langfuse
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLICEY"),
    secret_key=os.getenv("LANGFUSE_SECRETKEY"),
    host="https://us.cloud.langfuse.com"
)
langfuse = get_client()
langfuse_handler = CallbackHandler()

/tmp/ipykernel_153033/1823302574.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SerpAPIWrapper


#### Simple request to LLMs

In [ ]:
# Para GPT (OpenAI) # OPENAI_API_KEY must be a envar
llm = init_chat_model(
    model="gpt-4.1-mini", 
    #model_provider="openai",
    #model="gpt-5.5", 
    model_provider="openai",
    temperature=0.1,
    timeout=30,
    max_tokens=10
)

# Para Gemini (Google) # conseguir la GEMINI_API_KEY de Gemini Studio
#llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

response = llm.invoke("Porque hay lobos de color blanco?")
#print(response.content)
display(Markdown(response.content))

Los lobos pueden tener pelaje blanco debido a

In [9]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = llm.invoke(conversation)
display(Markdown(response.content))

J'adore créer des applications.

#### First agent

In [4]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = init_chat_model(
        model="gpt-4.1-mini", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
    )
agent = create_agent(
    model=llm, 
    tools=[get_weather],
    system_prompt="You are a professional weather assistant. Moreover, you always return your results in pretty markdown text"
)

response = agent.invoke(    {"messages": [{"role": "user", "content": "what is the weather in Arequipa"}]} )

In [5]:
final_message = response["messages"][-1].content
display(Markdown(final_message))
#print(final_message)

The weather in Arequipa is always sunny! If you need more detailed information or a forecast, feel free to ask.

In [35]:
response = agent.invoke(    
    {"messages": [{"role": "user", "content": "what is the weather in Lima"}]},
    config={"callbacks": [langfuse_handler]} 
)
final_message = response["messages"][-1].content
display(Markdown(final_message))

Failed to export span batch code: 401, reason: {"message":"Invalid credentials. Confirm that you've configured the correct host."}


## Weather in Lima

It’s **sunny** in **Lima** right now.

Failed to export span batch code: 401, reason: {"message":"Invalid credentials. Confirm that you've configured the correct host."}


#### Pre-defined tools

In [9]:
search = SerpAPIWrapper()
arxiv = ArxivAPIWrapper()

search_tool = Tool(
    name="web_search",
    description="Search the web for information",
    func=search.run
)

arxiv_tool = Tool(
    name="ArXiv",
    description="Search on ArXiv",
    func=arxiv.run
)

In [39]:
system_prompt = """
# Goal:
- Perform a deep research

# Tools
- **web_search**: Got web pages on the Web
- **ArXiv**: Got papers from ArXiv

# Intructions
- You must follow the user topic in the Web and ArXiv
- Then you must produce short review including citations in IEEE format.
- This is the structure of the review: (1) Summary, (2) Background and concepts, (3) Trends and challenges, (4) Conclusions, (5) References.

# Input
- User query

# Output
- Review and conclusions in markdown
"""

agent_deep_research = create_agent(
    model=llm, 
    tools=[search_tool, arxiv_tool],
    system_prompt=system_prompt
)


In [37]:
response = agent_deep_research.invoke(    {"messages": [{"role": "user", "content": "Busca los recientes avances en desarrollo de vacunas contra el Cancer"}]} )
final_message = response["messages"][-1].content
display(Markdown(final_message))

# Avances recientes en el desarrollo de vacunas contra el cáncer

## 1) Summary

Las vacunas contra el cáncer han pasado de ser una estrategia con resultados clínicos modestos a una plataforma central de inmunoterapia personalizada. El avance más relevante es la vacunación terapéutica basada en **neoantígenos**, especialmente mediante **ARNm personalizado**, diseñada a partir de la secuenciación del tumor de cada paciente. En melanoma resecado de alto riesgo, la vacuna individualizada **mRNA-4157/V940** combinada con pembrolizumab mostró una reducción clínicamente relevante del riesgo de recurrencia frente a pembrolizumab solo, impulsando ensayos fase III [1]. En cáncer de páncreas, la vacuna personalizada **autogene cevumeran** indujo respuestas de linfocitos T en aproximadamente la mitad de los pacientes tratados en un ensayo fase I, con señales de retraso en la recurrencia [2]. También avanzan vacunas “semiuniversales” dirigidas a mutaciones compartidas, como **KRAS G12D/G12R**, con la vacuna anfifílica **ELI-002** orientada a ganglios linfáticos [3].

En paralelo, se consolidan plataformas de **péptidos largos, células dendríticas, ARNm/lipopartículas, vectores virales, vacunas de antígenos compartidos** y combinaciones con inhibidores de puntos de control inmunitario. Los principales retos siguen siendo la selección precisa de antígenos, la heterogeneidad tumoral, la pérdida de HLA, el microambiente inmunosupresor, el coste y tiempo de fabricación, y la demostración de beneficio clínico en ensayos grandes.

---

## 2) Background and concepts

Las vacunas contra el cáncer pueden dividirse en dos grupos:

1. **Vacunas preventivas**: evitan infecciones oncogénicas. Las vacunas frente a **VPH** y **hepatitis B** han demostrado prevenir cánceres asociados a virus, como cáncer cervicouterino, anal, orofaríngeo y hepatocarcinoma [4].  
2. **Vacunas terapéuticas**: se administran a pacientes con cáncer para inducir o reforzar respuestas inmunes contra células tumorales. El ejemplo aprobado clásico es **sipuleucel-T** para cáncer de próstata metastásico resistente a castración, que prolongó la supervivencia global de forma modesta [5].

El concepto moderno más importante es el **neoantígeno tumoral**: una proteína mutada generada por alteraciones somáticas del tumor y ausente en tejidos normales. Al ser “no propia”, puede ser reconocida por linfocitos T con menor riesgo de autoinmunidad [6]. El proceso típico de una vacuna personalizada incluye:

- Secuenciación del tumor y tejido normal.
- Identificación de mutaciones somáticas.
- Tipificación HLA del paciente.
- Predicción bioinformática de péptidos presentados por HLA.
- Priorización de neoantígenos inmunogénicos.
- Fabricación de vacuna en plataforma ARNm, péptidos, células dendríticas o vectores.
- Administración junto con adyuvantes o inmunoterapia anti-PD-1/PD-L1.

Los estudios pioneros de Ott et al. y Sahin et al. demostraron que las vacunas personalizadas de neoantígenos podían inducir respuestas T policlonales en melanoma [7], [8]. Posteriormente, ensayos en glioblastoma mostraron que los linfocitos inducidos por vacunas podían infiltrar tumores cerebrales, aunque el beneficio clínico sigue siendo limitado por la inmunosupresión local [9].

---

## 3) Trends and challenges

### Tendencias principales

**a) ARNm personalizado como plataforma dominante**  
El éxito tecnológico de las vacunas de ARNm frente a COVID-19 aceleró la maduración de plataformas de ARNm oncológico. La vacuna **mRNA-4157/V940** codifica múltiples neoantígenos individuales en una formulación basada en nanopartículas lipídicas. En melanoma resecado, combinada con pembrolizumab, mostró mejor supervivencia libre de recurrencia que pembrolizumab solo [1]. Este resultado es uno de los hitos recientes más importantes porque sugiere que las vacunas pueden funcionar mejor en enfermedad mínima residual, cuando la carga tumoral es baja.

**b) Vacunas en cáncer de páncreas**  
El adenocarcinoma ductal pancreático es tradicionalmente poco inmunogénico. Sin embargo, Rojas et al. mostraron que una vacuna personalizada de ARNm, **autogene cevumeran**, podía inducir linfocitos T CD8+ frente a neoantígenos en pacientes operados, con asociación entre respuesta inmune y menor recurrencia temprana [2]. Aunque el ensayo fue pequeño, representa una señal relevante en un tumor de muy mal pronóstico.

**c) Antígenos compartidos: KRAS, telomerasa y antígenos inmunosupresores**  
Una alternativa a la personalización completa es dirigir vacunas contra mutaciones compartidas frecuentes. **ELI-002**, vacuna anfifílica dirigida a ganglios linfáticos contra KRAS mutado, mostró inmunogenicidad en cáncer pancreático y colorrectal con enfermedad mínima residual [3]. Este enfoque puede reducir costes y acelerar la producción, aunque solo aplica a subgrupos moleculares específicos.

**d) Combinación con checkpoint inhibitors**  
Las vacunas rara vez son suficientes por sí solas en tumores establecidos. La estrategia actual es combinarlas con anti-PD-1, anti-PD-L1, anti-CTLA-4, quimioterapia, radioterapia o agonistas innatos. La lógica es que la vacuna genera linfocitos T específicos, mientras que los inhibidores de puntos de control evitan su agotamiento funcional [1], [2].

**e) Bioinformática, IA y flujos reproducibles**  
La predicción de neoantígenos es un cuello de botella. Herramientas como pVACtools y flujos de trabajo recientes como **ImmunoNX** integran variantes somáticas, expresión de ARN, HLA y priorización inmunogénica para diseñar vacunas personalizadas de forma más reproducible [10], [11]. La tendencia es incorporar IA, inmunopeptidómica y datos de presentación real por HLA para reducir falsos positivos.

### Retos pendientes

- **Predicción imperfecta de neoantígenos**: muchos péptidos predichos no se presentan realmente ni inducen linfocitos funcionales.  
- **Heterogeneidad tumoral**: un neoantígeno puede estar presente solo en una subclona, permitiendo escape.  
- **Pérdida de presentación antigénica**: mutaciones o pérdida de HLA, β2-microglobulina o maquinaria de procesamiento antigénico reducen eficacia.  
- **Microambiente tumoral inmunosupresor**: TGF-β, células T reguladoras, macrófagos M2 y células mieloides supresoras limitan la respuesta.  
- **Tiempo de fabricación**: las vacunas personalizadas deben producirse en semanas, idealmente antes de la recurrencia.  
- **Coste y escalabilidad**: la personalización exige infraestructura genómica, bioinformática, GMP y control de calidad.  
- **Validación clínica**: muchos ensayos muestran inmunogenicidad, pero aún falta demostrar supervivencia global en estudios fase III para la mayoría de plataformas.

---

## 4) Conclusions

Los avances más sólidos se concentran en vacunas terapéuticas personalizadas de neoantígenos, especialmente de ARNm, aplicadas en contexto adyuvante o de enfermedad mínima residual. Melanoma y cáncer de páncreas son los ejemplos recientes más visibles, mientras que vacunas contra mutaciones compartidas como KRAS podrían ofrecer una vía más escalable. El futuro probablemente combinará **secuenciación tumoral rápida, predicción de neoantígenos asistida por IA, plataformas ARNm o peptídicas optimizadas, y bloqueo de checkpoints**.

A corto plazo, el campo debe demostrar que la fuerte inmunogenicidad observada se traduce en beneficios robustos de supervivencia y que la fabricación personalizada puede ser rápida, segura y coste-efectiva. Si los ensayos fase III confirman los resultados iniciales, las vacunas contra el cáncer podrían convertirse en una nueva columna de la oncología de precisión.

---

## 5) References

[1] J. S. Weber *et al*., “Individualised neoantigen therapy mRNA-4157/V940 plus pembrolizumab versus pembrolizumab monotherapy in resected melanoma: KEYNOTE-942, a randomised phase 2b study,” *The Lancet*, 2024.

[2] L. A. Rojas *et al*., “Personalized RNA neoantigen vaccines stimulate T cells in pancreatic cancer,” *Nature*, vol. 618, pp. 144–150, 2023.

[3] S. Pant *et al*., “Lymph-node-targeted, mKRAS-specific amphiphile vaccine in pancreatic and colorectal cancer: the phase 1 AMPLIFY-201 trial,” *Nature Medicine*, vol. 30, pp. 531–542, 2024.

[4] World Health Organization, “Human papillomavirus vaccines: WHO position paper,” *Weekly Epidemiological Record*, 2022.

[5] P. W. Kantoff *et al*., “Sipuleucel-T immunotherapy for castration-resistant prostate cancer,” *New England Journal of Medicine*, vol. 363, no. 5, pp. 411–422, 2010.

[6] T. N. Schumacher and R. D. Schreiber, “Neoantigens in cancer immunotherapy,” *Science*, vol. 348, no. 6230, pp. 69–74, 2015.

[7] P. A. Ott *et al*., “An immunogenic personal neoantigen vaccine for patients with melanoma,” *Nature*, vol. 547, pp. 217–221, 2017.

[8] U. Sahin *et al*., “Personalized RNA mutanome vaccines mobilize poly-specific therapeutic immunity against cancer,” *Nature*, vol. 547, pp. 222–226, 2017.

[9] D. B. Keskin *et al*., “Neoantigen vaccine generates intratumoral T cell responses in phase Ib glioblastoma trial,” *Nature*, vol. 565, pp. 234–239, 2019.

[10] J. Hundal *et al*., “pVACtools: A computational toolkit to identify and visualize cancer neoantigens,” *Cancer Immunology Research*, vol. 8, no. 3, pp. 409–420, 2020.

[11] K. Singhal *et al*., “ImmunoNX: a robust bioinformatics workflow to support personalized neoantigen vaccine trials,” *arXiv preprint*, 2025.

### Exercise

- Choose another two tools from this link: https://docs.langchain.com/oss/python/integrations/tools
- Improve the agent_deep_research including the tools choosen. Also improve the system prompt according to your personal style.
- Analyze the trace in LangFuse/LangSmith and evaluate using other LLMs

## MCP

#### Using public MCP servers

MCP requieres Async invocation

In [3]:
import nest_asyncio
nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

True

In [4]:
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/vicente/cursos/agents/src"],
        "transport": "stdio",
    }
}
# El servidor se levanta con el comando npx localmente, con stdio siempre es localmente

llm = init_chat_model(
        model="gpt-5.5", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
)       

client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()
print(f"Tools loaded: {[t.name for t in tools]}")

agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a file system assistant."
)

async def run_agent(query: str):     
    result = await agent.ainvoke({"messages": [{"role": "user", "content": query}]} )
    return result["messages"][-1].content

Tools loaded: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


In [5]:
response = await run_agent("List all files in the current directory")
print(response)

Files in the current directory:

- `.env`
- `.env_sample`
- `1_prompts_techniques.ipynb`
- `2_hello_world_agent.ipynb`
- `3_memory_and_reflection.ipynb`
- `4_msa_agents.ipynb`
- `5-feedback.ipynb`
- `5_react_weather_agent.ipynb`
- `6-MSA_landgraph.ipynb`
- `7-MSA_landgchain.ipynb`
- `8-MSA-collaborative_learning.ipynb`
- `FAISS.ipynb`
- `_3-feedback.ipynb`
- `_langchain-logs.ipynb`
- `_langchain_hello.ipynb`
- `_reflection_agent.ipynb`
- `airbnb.csv`
- `bert_finetuning_tutorial.ipynb`
- `faiss_pca_visualization.png`
- `faiss_tsne_visualization.png`
- `function_calling.mermaid`
- `function_calling_basic.mermaid`
- `gradient_descent.html`
- `litellm_proxy_examples.ipynb`
- `llm_finetuning_lora.ipynb`
- `mcp_architecture.mermaid`
- `mcp_flow.mermaid`
- `mcp_test_file.txt`
- `mcp_test_file_french.txt`
- `neuron_simulator.html`
- `requirements.txt`
- `server_mcp.py`
- `server_mcp_http.py`
- `tokenizer_embedding.html`
- `utils.py`

Directories:

- `__pycache__`
- `_mcp`
- `adk`
- `mcp_exampl

In [6]:
response = await run_agent("Read the contents of mcp_test_file.txt, then create a new file with the same content in French, The new file name must be mcp_test_file_french.txt")
print(response)


Done.


#### Example with more MCP servers

In [7]:
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/vicente/cursos/agents/src"],
        "transport": "stdio",
    },
    "airbnb": {
      "command": "npx",
      "args": [
        "-y",
        "@openbnb/mcp-server-airbnb",
        "--ignore-robots-txt"
      ],
      "transport": "stdio"
    }
}

llm = init_chat_model(
        model="gpt-5.5", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
)       

client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()
print(f"Tools loaded: {[t.name for t in tools]}")

agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a file system assistant."
)

async def run_agent(query: str):     
    result = await agent.ainvoke({"messages": [{"role": "user", "content": query}]} )
    return result["messages"][-1].content

Tools loaded: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'airbnb_search', 'airbnb_listing_details']


In [8]:
response = await run_agent("Search places (airbnb) in Arequipa, Peru from 10/05/2026 to 20/05/2026. Then create a file csv name airbnb.csv with a summary of your results")
print(response)

Done — I searched Airbnb for Arequipa, Peru from 2026-05-10 to 2026-05-20 and created the CSV file:

`/home/vicente/cursos/agents/src/airbnb.csv`

It contains a summary of 18 Airbnb search results with listing ID, name, URL, badge, details, rating, price, price details, latitude, and longitude.


#### Exersice

Create an agent using MCP tools, choose tools from: https://mcpservers.org/all